# Batch encoding

- The output of a tokenizer isn’t a simple Python dictionary; what we get is actually a special `BatchEncoding` object.
- It’s a **subclass of a dictionary** (which is why we were able to index into that result without any problem before), but with additional methods that are mostly used by fast tokenizers.

- Besides their **parallelization** capabilities, the key functionality of fast tokenizers is that they always **keep track of the original span** of texts the final tokens come from — a feature we call __offset mapping__.
    - This in turn unlocks features like
        - mapping each word to the tokens it generated
        - or mapping each character of the original text to the token it’s inside, and vice versa.

Let’s take a look at an example:

In [ ]:
from transformers import AutoTokenizer
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
print(f"is bert tokenizer fast: {bert_tokenizer.is_fast}")
roberta_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
print(f"is roberta tokenizer fast: {roberta_tokenizer.is_fast}")

## Word <--> Token <--> Sentence

In [ ]:
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
encoding = bert_tokenizer(example)
print(type(encoding))
print("====================")
print(f"is fast: {encoding.is_fast}")
print("====================")
print(f"tokens: {encoding.tokens()}")
print("====================")
print(f"word ids: {encoding.word_ids()}")
print("====================")
print(f"sentence ids: {encoding.token_type_ids}")

- The tokenizer’s special tokens `[CLS]` and `[SEP]` are mapped to `None`
- Each token is mapped to the word it originates from.
    - This is especially useful to determine if a token is at the start of a word 
    - or if two tokens are in the same word.
    - this method works for **any** type of tokenizer as long as it’s a **fast** one.
- We can use the `##` prefix to map tokens to words, but it only works for **BERT-like** tokenizers;

==============
- The notion of what a word is complicated.
- For instance, does “I’ll” (a contraction of “I will”) count as one or two words?
    - It actually depends on the tokenizer and the pre-tokenization operation it applies.
    - Some tokenizers just split on spaces, so they will consider this as one word.
    - Others use punctuation on top of spaces, so will consider it two words.
- We can see the difference between `bert` and `roberta` tokenizer for `"81s"`

In [ ]:
example = "81s"
encoding = bert_tokenizer(example)
print(f"bert tokens: {encoding.tokens()}")
print(f"bert word ids: {encoding.word_ids()}")
print("====================")
encoding = roberta_tokenizer(example)
print(f"roberta tokens: {encoding.tokens()}")
print(f"roberta word ids: {encoding.word_ids()}")

## Token <--> Char <--> Word

In [ ]:
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
encoding = bert_tokenizer(example)
print(f"words: {encoding.word_ids()}")
print("=====================")
print(f"tokens: {encoding.tokens()}")
print("=====================")
word_id = 3
s, e = encoding.word_to_chars(word_id)
print(f"word number {word_id}: {example[s:e]}")
print("=====================")
token_id = 5
s,e = encoding.token_to_chars(token_id)
print(f"token number {token_id}: {example[s:e]}")
print("=====================")
char_id = 30
print(f"char {char_id} belongs to word {encoding.char_to_word(char_id)} and token {encoding.char_to_token(char_id)}")

# Inside the token-classification pipeline

## Pipeline inference example

The model used by default is [`dbmdz/bert-large-cased-finetuned-conll03-english`](https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english)

In [ ]:
from pprint import pprint
from transformers import pipeline
token_classifier = pipeline("token-classification")
result = token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")
for line in result: print(line)

We can also ask the pipeline to group together the tokens that correspond to the same entity

- The `aggregation_strategy` picked will change the scores computed for each grouped entity.
- With `"simple"` the score is just the mean of the scores of each token in the given entity
    - for instance, the score of “Sylvain” is the mean of the scores we saw in the previous example for the tokens `S`, `##yl`, `##va`, and `##in`.
- Other strategies available are:
    - `"first"`, where the score of each entity is the score of the first token of that entity
    - `"max"`, where the score of each entity is the maximum score of the tokens in that entity
    - `"average"`, where the score of each entity is the average of the scores of the words composing that entity
        - for “Sylvain” there would be no difference from the `"simple"` strategy,
        - but “Hugging Face” would have a score of 0.9819, the average of the scores for “Hugging”, 0.975, and “Face”, 0.98879

In [ ]:
token_classifier = pipeline("token-classification", aggregation_strategy="simple")
agg_result = token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")
for line in agg_result: print(line)

## Inference without pipeline

In [ ]:
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
inputs = tokenizer(
    example,
    return_tensors="pt",
    return_offsets_mapping=True
)


offset_mapping = inputs.pop("offset_mapping")
outputs = model(**inputs)
print("======================")
print(inputs["input_ids"].shape)
print("======================")
probabilities = F.softmax(outputs.logits, dim=-1)[0].data
print(outputs.logits.shape)
print(probabilities.shape)

predictions = outputs.logits.argmax(dim=-1)[0].tolist()
offset_mapping = offset_mapping.squeeze().tolist()

print("======================")
print(model.config.id2label)
results = []
print("======================")
for i, (t, p) in enumerate(zip(inputs.tokens(), predictions)):
    if p != 0:
        results.append(
            {
                "entity": model.config.id2label[p],
                "score": probabilities[i, p],
                "index": i,
                "word": t,
                "start": offset_mapping[i][0],
                "end": offset_mapping[i][1]
            }
        )
for line in results: print(line)

- In the current example we would expect our model to classify the token `S` as `B-PER` (beginning of a person entity) and the tokens `##yl`, `##va` and `##in` as `I-PER` (inside a person entity).
- You might think the model was wrong in this case as it gave the label `I-PER` to all four of these tokens, but that’s not entirely true.
- There are actually two formats for those `B-` and `I-` labels: _IOB1_ and _IOB2_.
- In the _IOB1_ format , the labels beginning with `B-` are only ever used to separate two adjacent entities of the same type.
- The model we are using was fine-tuned on a dataset using that format, which is why it assigns the label `I-PER` to the `S` token.

### Grouping entities

In [ ]:
results = []
prev_class = None

curr_start = None
curr_end = None
score_list = []
print("======================")
for i, (t, p) in enumerate(zip(inputs.tokens(), predictions)):
    label = model.config.id2label[p]
    if label.startswith(("O", "B")): # reset
        if score_list:
            label_list = list(set(label_list))
            if len(label_list) > 1: raise ValueError("multiple labels for an entity")
            results.append(
                {
                    "entity_group": label_list[0],
                    "score": sum(score_list)/len(score_list),
                    "word": example[curr_start: curr_end],
                    "start": curr_start,
                    "end": curr_end,
                }
            )
        curr_start = None
        score_list = []
        label_list = []
    if not label.startswith("O"):
        if not curr_start: curr_start, curr_end = offset_mapping[i]
        else: curr_end = offset_mapping[i][1]
        score_list.append(probabilities[i, p])
        label_list.append(label.split("-")[-1])
for line in results: print(line)